# 🎯 Smart Object Tracking + Anomaly Alert System
### Track Cars · Trucks · Buses · Humans across 5 Video Categories
---
> **Detects:** People, Vehicles across Traffic · Crowd · Parking · Metro · Street footage  
> **Alerts on:** Overcrowding · Wrong-way driving · Abandoned objects · Sudden stops · Fights · Trespassing  
> **Engine:** YOLOv8 Detection + ByteTrack Multi-Object Tracking  

---
```
CELL 1  →  Install Libraries
CELL 2  →  Verify Everything Works
CELL 3  →  Configuration (paths, thresholds, classes)
CELL 4  →  Core Detection + Tracking Engine
CELL 5  →  Anomaly Detection Rules (per video category)
CELL 6  →  Alert System (visual + sound + log)
CELL 7  →  Run on Traffic Video
CELL 8  →  Run on Crowd / Metro Video
CELL 9  →  Run on Parking Lot Video
CELL 10 →  Run on Street / Pedestrian Video
CELL 11 →  Run on Custom / Your Own Video
CELL 12 →  Full Analytics Dashboard
```

---
## ⚙️ CELL 1 — Install All Libraries
Run once. Takes about 2–4 minutes.

In [ ]:
import sys, subprocess

packages = [
    'ultralytics',        # YOLOv8 — detection + built-in ByteTrack
    'opencv-python',      # Video read/write/display
    'numpy',
    'pandas',
    'matplotlib',
    'seaborn',
    'Pillow',
    'scipy',
    'ipywidgets',
    'tqdm',
]

print('🚀 Installing packages...\n')
for pkg in packages:
    print(f'   📦  {pkg:<22}', end=' ')
    r = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', pkg, '--quiet', '--upgrade'],
        capture_output=True, text=True
    )
    print('✅' if r.returncode == 0 else f'❌  {r.stderr[:60]}')

print('\n✅ All libraries installed! Run Cell 2 to verify.')

---
## 🔍 CELL 2 — Verify Libraries + Auto-Download YOLOv8 Model

In [ ]:
print('=' * 55)
print('   🔍 LIBRARY + MODEL VERIFICATION')
print('=' * 55)

checks = []

try:
    import cv2;         checks.append(('OpenCV',      cv2.__version__,        True))
except Exception as e: checks.append(('OpenCV',      str(e)[:50],            False))

try:
    import numpy as np; checks.append(('NumPy',       np.__version__,         True))
except Exception as e: checks.append(('NumPy',       str(e)[:50],            False))

try:
    import pandas as pd;checks.append(('Pandas',      pd.__version__,         True))
except Exception as e: checks.append(('Pandas',      str(e)[:50],            False))

try:
    import matplotlib;  checks.append(('Matplotlib',  matplotlib.__version__, True))
except Exception as e: checks.append(('Matplotlib',  str(e)[:50],            False))

try:
    from ultralytics import YOLO
    import ultralytics
    checks.append(('Ultralytics',  ultralytics.__version__, True))
except Exception as e:
    checks.append(('Ultralytics',  str(e)[:50],             False))

all_ok = True
for name, ver, ok in checks:
    icon = '✅' if ok else '❌'
    print(f'  {icon}  {name:<18}  {"v"+ver if ok else ver}')
    if not ok: all_ok = False

print('=' * 55)

if all_ok:
    print('\n📥 Downloading YOLOv8n model (auto, ~6 MB)...')
    from ultralytics import YOLO
    model_test = YOLO('yolov8n.pt')   # Downloads automatically on first run
    print('✅ YOLOv8n model ready!')
    print('\n🎉 EVERYTHING WORKS! Proceed to Cell 3.')
else:
    print('\n⚠️  Some libraries failed. Re-run Cell 1 then try again.')

---
## ⚙️ CELL 3 — Configuration: Paths, Classes & Anomaly Thresholds
**Edit this cell to point to your videos. All other cells read from here.**

In [ ]:
import os
import numpy as np

# ══════════════════════════════════════════════════════════════
#  📁  YOUR 5 VIDEO FILES — point each to a real .mp4 / .avi
# ══════════════════════════════════════════════════════════════
VIDEO_PATHS = {
    'traffic':   r'D:\AI FOR SOCIAL GOOD 1\videos\traffic.mp4',
    'crowd':     r'D:\AI FOR SOCIAL GOOD 1\videos\crowd.mp4',
    'parking':   r'D:\AI FOR SOCIAL GOOD 1\videos\parking.mp4',
    'metro':     r'D:\AI FOR SOCIAL GOOD 1\videos\metro.mp4',
    'street':    r'D:\AI FOR SOCIAL GOOD 1\videos\street.mp4',
}

# ══════════════════════════════════════════════════════════════
#  📂  OUTPUT FOLDER — annotated videos + logs saved here
# ══════════════════════════════════════════════════════════════
OUTPUT_DIR = r'D:\AI FOR SOCIAL GOOD 1\tracking_output'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ══════════════════════════════════════════════════════════════
#  🏷️  COCO CLASSES WE CARE ABOUT (YOLOv8 COCO class IDs)
# ══════════════════════════════════════════════════════════════
TRACK_CLASSES = {
    0:  {'name': 'Person',      'color': (0,   255, 100), 'category': 'human'},
    2:  {'name': 'Car',         'color': (255, 165,   0), 'category': 'vehicle'},
    3:  {'name': 'Motorcycle',  'color': (255,  50, 150), 'category': 'vehicle'},
    5:  {'name': 'Bus',         'color': (0,   150, 255), 'category': 'vehicle'},
    7:  {'name': 'Truck',       'color': (180,   0, 255), 'category': 'vehicle'},
    24: {'name': 'Backpack',    'color': (255, 255,   0), 'category': 'object'},
    26: {'name': 'Handbag',     'color': (255, 200,   0), 'category': 'object'},
    28: {'name': 'Suitcase',    'color': (200, 255,   0), 'category': 'object'},
}
TRACK_CLASS_IDS = list(TRACK_CLASSES.keys())

# ══════════════════════════════════════════════════════════════
#  🚨  ANOMALY THRESHOLDS (tune per camera/location)
# ══════════════════════════════════════════════════════════════
ANOMALY_RULES = {

    # --- TRAFFIC SCENE ---
    'traffic': {
        'max_vehicles_in_frame':  25,    # 🔴 Traffic jam if exceeded
        'min_vehicle_speed_px':    1.5,  # 🔴 Vehicle stopped if below this (px/frame)
        'stopped_frames_thresh':  45,    # 🔴 Flag stopped vehicle after N frames
        'pedestrian_in_road':     True,  # 🔴 Alert if person detected in vehicle zone
        'wrong_way_detect':       True,  # 🔴 Movement against dominant flow
    },

    # --- CROWD / EVENT SCENE ---
    'crowd': {
        'max_persons_in_frame':   40,    # 🔴 Overcrowding
        'min_persons_in_frame':    2,    # 🔴 Abnormally empty (scene should be busy)
        'crowd_density_thresh':   0.15,  # 🔴 Fraction of frame covered by people
        'running_speed_px':       12,    # 🔴 Person running (px/frame) — possible fight
        'sudden_disperse':        True,  # 🔴 Count drops >50% in 10 frames → panic
    },

    # --- PARKING LOT ---
    'parking': {
        'max_persons_in_frame':    5,    # 🔴 Too many people in lot
        'loitering_frames_thresh': 120,  # 🔴 Person standing still for N frames
        'abandoned_object_frames': 90,   # 🔴 Bag/suitcase without person nearby
        'vehicle_speeding_px':     15,   # 🔴 Vehicle moving too fast in lot
    },

    # --- METRO / STATION ---
    'metro': {
        'max_persons_in_frame':   60,    # 🔴 Dangerous platform density
        'platform_edge_zone_pct': 0.90,  # 🔴 Y > 90% of frame = near track edge
        'running_speed_px':       10,    # 🔴 Running on platform
        'sudden_disperse':        True,
        'fall_aspect_thresh':     2.5,   # 🔴 bbox width/height > 2.5 = person fallen
    },

    # --- STREET / PEDESTRIAN ---
    'street': {
        'max_persons_in_frame':   20,
        'loitering_frames_thresh': 150,
        'vehicle_in_ped_zone':    True,  # 🔴 Vehicle enters pedestrian-only area
        'running_speed_px':        8,
        'abandoned_object_frames': 100,
    },
}

# ══════════════════════════════════════════════════════════════
#  🎬  PROCESSING SETTINGS
# ══════════════════════════════════════════════════════════════
YOLO_MODEL       = 'yolov8n.pt'   # n=nano(fast) | s=small | m=medium | l=large
CONF_THRESHOLD   = 0.45           # Detection confidence (0–1)
IOU_THRESHOLD    = 0.5            # NMS IOU threshold
PROCESS_EVERY_N  = 2              # Process every Nth frame (2 = skip every other)
MAX_FRAMES       = 1200           # Max frames per video (None = full video)
SHOW_LIVE        = False          # Set True to open live window (slow in notebooks)

# ══════════════════════════════════════════════════════════════
#  🎨  VISUALIZATION SETTINGS
# ══════════════════════════════════════════════════════════════
ALERT_COLOR      = (0,   0, 255)  # Red  — alert boxes
TRAIL_LENGTH     = 30             # Frames to keep movement trails
FONT             = cv2.FONT_HERSHEY_SIMPLEX if 'cv2' in dir() else None

# ── Verify videos ──
import cv2
print('📁 Video File Check:')
print('─' * 55)
for scene, path in VIDEO_PATHS.items():
    if os.path.exists(path):
        cap = cv2.VideoCapture(path)
        fps = cap.get(cv2.CAP_PROP_FPS)
        fc  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        w   = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        h   = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        cap.release()
        print(f'  ✅  {scene:<10} {w}×{h}  {fps:.0f}fps  {fc} frames  →  {os.path.basename(path)}')
    else:
        print(f'  ⚠️   {scene:<10} FILE NOT FOUND → {path}')
        print(f'       (Place your video at that path or update VIDEO_PATHS above)')
print('─' * 55)
print('\n✅ Configuration loaded. Proceed to Cell 4.')

---
## 🧠 CELL 4 — Core Tracking Engine
All the heavy lifting: detection, tracking, trail drawing, counter overlays.  
**Just run this cell — no edits needed.**

In [ ]:
import cv2
import numpy as np
import pandas as pd
from collections import defaultdict, deque
from ultralytics import YOLO
import time, os

# ─────────────────────────────────────────────────────
# Object tracker state (persists across frames)
# ─────────────────────────────────────────────────────
class ObjectTracker:
    """
    Wraps YOLOv8's built-in ByteTrack and adds:
    - Per-object movement trails
    - Speed estimation (pixels/frame)
    - Stopped-frame counter per track
    - Class category grouping
    """
    def __init__(self):
        self.trails         = defaultdict(lambda: deque(maxlen=TRAIL_LENGTH))
        self.speeds         = defaultdict(float)     # px/frame
        self.stopped_frames = defaultdict(int)       # frames not moving
        self.still_frames   = defaultdict(int)       # for loitering
        self.last_center    = {}                     # last known center
        self.class_history  = {}                     # track_id → class_id
        self.first_seen     = {}                     # track_id → frame number
        self.last_bbox      = {}                     # track_id → (x1,y1,x2,y2)
        self.frame_count    = 0

    def update(self, results, frame_idx):
        """
        Parse YOLO tracking results for one frame.
        Returns list of dicts with track info.
        """
        self.frame_count = frame_idx
        active_tracks = []

        if results[0].boxes is None:
            return active_tracks

        boxes  = results[0].boxes
        if boxes.id is None:
            return active_tracks

        ids    = boxes.id.cpu().numpy().astype(int)
        clses  = boxes.cls.cpu().numpy().astype(int)
        confs  = boxes.conf.cpu().numpy()
        xyxys  = boxes.xyxy.cpu().numpy()

        for tid, cls_id, conf, xyxy in zip(ids, clses, confs, xyxys):
            if cls_id not in TRACK_CLASS_IDS:
                continue

            x1, y1, x2, y2 = map(int, xyxy)
            cx = (x1 + x2) // 2
            cy = (y1 + y2) // 2
            w  = x2 - x1
            h  = y2 - y1

            # Speed calculation
            if tid in self.last_center:
                dx = cx - self.last_center[tid][0]
                dy = cy - self.last_center[tid][1]
                speed = (dx**2 + dy**2) ** 0.5
                self.speeds[tid] = round(speed, 2)
                if speed < 1.2:
                    self.stopped_frames[tid] += 1
                    self.still_frames[tid]   += 1
                else:
                    self.stopped_frames[tid] = 0
                    self.still_frames[tid]   = 0
            else:
                self.speeds[tid]         = 0
                self.stopped_frames[tid] = 0
                self.still_frames[tid]   = 0
                self.first_seen[tid]     = frame_idx

            self.last_center[tid]  = (cx, cy)
            self.class_history[tid]= cls_id
            self.last_bbox[tid]    = (x1, y1, x2, y2)
            self.trails[tid].append((cx, cy))

            active_tracks.append({
                'id':       tid,
                'cls_id':   cls_id,
                'name':     TRACK_CLASSES[cls_id]['name'],
                'color':    TRACK_CLASSES[cls_id]['color'],
                'category': TRACK_CLASSES[cls_id]['category'],
                'conf':     round(float(conf), 2),
                'bbox':     (x1, y1, x2, y2),
                'center':   (cx, cy),
                'speed':    self.speeds[tid],
                'stopped_frames': self.stopped_frames[tid],
                'still_frames':   self.still_frames[tid],
                'width':    w,
                'height':   h,
                'aspect':   round(w / max(h, 1), 2),
            })

        return active_tracks


# ─────────────────────────────────────────────────────
# Drawing utilities
# ─────────────────────────────────────────────────────
def draw_box(frame, track, alert=False, alert_msg=''):
    """Draw bounding box, ID label, speed, and optional alert marker."""
    x1, y1, x2, y2 = track['bbox']
    color  = ALERT_COLOR if alert else track['color']
    thick  = 3 if alert else 2

    # Box
    cv2.rectangle(frame, (x1, y1), (x2, y2), color, thick)

    # Corner accents for alert
    if alert:
        corner = 15
        cv2.line(frame, (x1, y1), (x1+corner, y1), (0, 0, 255), 3)
        cv2.line(frame, (x1, y1), (x1, y1+corner), (0, 0, 255), 3)
        cv2.line(frame, (x2, y2), (x2-corner, y2), (0, 0, 255), 3)
        cv2.line(frame, (x2, y2), (x2, y2-corner), (0, 0, 255), 3)

    # Label background
    label      = f"{track['name']} #{track['id']}  {track['speed']:.1f}px/f"
    (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.45, 1)
    label_y    = max(y1 - 5, th + 4)
    cv2.rectangle(frame, (x1, label_y-th-4), (x1+tw+4, label_y+2), color, -1)
    cv2.putText(frame, label, (x1+2, label_y-1),
                cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0, 0, 0), 1, cv2.LINE_AA)

    # Alert message above box
    if alert and alert_msg:
        cv2.putText(frame, f'⚠ {alert_msg}', (x1, y1 - 20),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 0, 255), 2, cv2.LINE_AA)


def draw_trails(frame, tracker):
    """Draw motion trails for every tracked object."""
    for tid, trail in tracker.trails.items():
        pts = list(trail)
        cls_id = tracker.class_history.get(tid)
        color  = TRACK_CLASSES[cls_id]['color'] if cls_id in TRACK_CLASSES else (200,200,200)
        for i in range(1, len(pts)):
            alpha = int(255 * i / len(pts))
            c     = tuple(int(x * alpha / 255) for x in color)
            cv2.line(frame, pts[i-1], pts[i], c, 2)


def draw_hud(frame, counts, alerts, fps_disp, scene, frame_idx):
    """Draw the top-left HUD panel with live counts."""
    H, W = frame.shape[:2]

    # Semi-transparent panel
    overlay = frame.copy()
    cv2.rectangle(overlay, (5, 5), (260, 160), (20, 20, 20), -1)
    cv2.addWeighted(overlay, 0.65, frame, 0.35, 0, frame)

    cv2.putText(frame, f'SCENE: {scene.upper()}', (12, 24),
                cv2.FONT_HERSHEY_SIMPLEX, 0.55, (100, 220, 255), 2)
    cv2.putText(frame, f'Frame: {frame_idx}   FPS: {fps_disp:.1f}', (12, 44),
                cv2.FONT_HERSHEY_SIMPLEX, 0.42, (180, 180, 180), 1)

    y = 65
    category_icon = {'human': '🧍', 'vehicle': '🚗', 'object': '🎒'}
    color_map     = {'human': (0,255,100), 'vehicle': (0,165,255), 'object': (0,255,255)}
    for cat, cnt in counts.items():
        clr = color_map.get(cat, (200,200,200))
        cv2.putText(frame, f'{cat.capitalize():10s}: {cnt}', (12, y),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.48, clr, 1)
        y += 20

    # Alert count badge
    if alerts:
        cv2.putText(frame, f'ALERTS: {len(alerts)}', (12, y+5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.52, (0, 0, 255), 2)
        # Flashing red border when alert
        if (frame_idx // 10) % 2 == 0:
            cv2.rectangle(frame, (0, 0), (W-1, H-1), (0, 0, 255), 4)


def draw_alert_banner(frame, alerts):
    """Draw a scrolling alert list at the bottom of the frame."""
    if not alerts:
        return
    H, W = frame.shape[:2]
    overlay = frame.copy()
    banner_h = min(len(alerts), 4) * 22 + 8
    cv2.rectangle(overlay, (0, H - banner_h - 5), (W, H), (10, 10, 60), -1)
    cv2.addWeighted(overlay, 0.75, frame, 0.25, 0, frame)
    for i, alrt in enumerate(alerts[-4:]):   # show last 4
        cv2.putText(frame, f'  ⚠  {alrt}',
                    (8, H - banner_h + i*22 + 18),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.44,
                    (0, 100, 255), 1, cv2.LINE_AA)


print('✅ Tracking engine loaded — ObjectTracker + all drawing helpers ready.')

---
## 🚨 CELL 5 — Anomaly Detection Rules Engine
Each scene type has its own rule set. This cell contains all the logic.

In [ ]:
from collections import deque

class AnomalyDetector:
    """
    Scene-aware anomaly detection.
    Call detect(tracks, frame, scene, frame_idx) every frame.
    Returns list of alert strings for that frame.
    """
    def __init__(self):
        self.person_count_history = deque(maxlen=30)  # rolling window
        self.alert_log            = []                # full session log
        self.last_alert_frame     = defaultdict(lambda: -999)  # cooldown
        self.COOLDOWN             = 30  # min frames between same alert type

    def _can_alert(self, alert_type, frame_idx):
        """Prevent the same alert from spamming every frame."""
        if frame_idx - self.last_alert_frame[alert_type] >= self.COOLDOWN:
            self.last_alert_frame[alert_type] = frame_idx
            return True
        return False

    def detect(self, tracks, tracker, frame, scene, frame_idx, frame_h, frame_w):
        """
        Main detection function.
        Returns (list_of_alert_strings, dict_of_track_id_to_alert_msg)
        """
        rules      = ANOMALY_RULES.get(scene, {})
        alerts     = []   # frame-level alert messages
        track_alrt = {}   # track_id → alert label for bounding box

        persons  = [t for t in tracks if t['category'] == 'human']
        vehicles = [t for t in tracks if t['category'] == 'vehicle']
        objects  = [t for t in tracks if t['category'] == 'object']

        n_persons  = len(persons)
        n_vehicles = len(vehicles)

        self.person_count_history.append(n_persons)

        # ══════════════════════════════════════════
        # 1. OVERCROWDING  (crowd / metro / street)
        # ══════════════════════════════════════════
        max_p = rules.get('max_persons_in_frame')
        if max_p and n_persons > max_p:
            if self._can_alert('overcrowd', frame_idx):
                msg = f'OVERCROWDING — {n_persons} people (limit {max_p})'
                alerts.append(msg)
                self._log(scene, frame_idx, 'OVERCROWDING', msg)
                # Flag all people
                for t in persons:
                    track_alrt[t['id']] = 'CROWD'

        # ══════════════════════════════════════════
        # 2. TRAFFIC JAM  (traffic)
        # ══════════════════════════════════════════
        max_v = rules.get('max_vehicles_in_frame')
        if max_v and n_vehicles > max_v:
            if self._can_alert('traffic_jam', frame_idx):
                msg = f'TRAFFIC JAM — {n_vehicles} vehicles'
                alerts.append(msg)
                self._log(scene, frame_idx, 'TRAFFIC_JAM', msg)

        # ══════════════════════════════════════════
        # 3. STOPPED VEHICLE  (traffic)
        # ══════════════════════════════════════════
        stopped_thresh = rules.get('stopped_frames_thresh', 9999)
        for t in vehicles:
            if t['stopped_frames'] >= stopped_thresh:
                alert_key = f'stopped_{t["id"]}'
                if self._can_alert(alert_key, frame_idx):
                    msg = f'STOPPED VEHICLE #{t["id"]} ({t["name"]}) — {t["stopped_frames"]}f'
                    alerts.append(msg)
                    self._log(scene, frame_idx, 'STOPPED_VEHICLE', msg)
                track_alrt[t['id']] = 'STOPPED'

        # ══════════════════════════════════════════
        # 4. RUNNING / FAST MOVEMENT  (crowd/metro/street)
        # ══════════════════════════════════════════
        run_thresh = rules.get('running_speed_px', 9999)
        for t in persons:
            if t['speed'] >= run_thresh:
                alert_key = f'run_{t["id"]}'
                if self._can_alert(alert_key, frame_idx):
                    msg = f'PERSON RUNNING #{t["id"]} speed={t["speed"]:.1f}px/f'
                    alerts.append(msg)
                    self._log(scene, frame_idx, 'RUNNING', msg)
                track_alrt[t['id']] = 'RUNNING'

        # ══════════════════════════════════════════
        # 5. LOITERING  (parking / street)
        # ══════════════════════════════════════════
        loiter_thresh = rules.get('loitering_frames_thresh', 9999)
        for t in persons:
            if tracker.still_frames[t['id']] >= loiter_thresh:
                alert_key = f'loiter_{t["id"]}'
                if self._can_alert(alert_key, frame_idx):
                    secs = tracker.still_frames[t['id']] / 25  # approx @25fps
                    msg  = f'LOITERING #{t["id"]} — stationary {secs:.0f}s'
                    alerts.append(msg)
                    self._log(scene, frame_idx, 'LOITERING', msg)
                track_alrt[t['id']] = 'LOITERING'

        # ══════════════════════════════════════════
        # 6. ABANDONED OBJECT  (parking / street)
        # ══════════════════════════════════════════
        aband_thresh = rules.get('abandoned_object_frames', 9999)
        for obj in objects:
            if tracker.still_frames[obj['id']] >= aband_thresh:
                # Check if any person is nearby
                ox, oy = obj['center']
                nearby = any(
                    abs(p['center'][0]-ox) < 80 and abs(p['center'][1]-oy) < 80
                    for p in persons
                )
                if not nearby:
                    alert_key = f'aband_{obj["id"]}'
                    if self._can_alert(alert_key, frame_idx):
                        msg = f'ABANDONED {obj["name"]} #{obj["id"]}'
                        alerts.append(msg)
                        self._log(scene, frame_idx, 'ABANDONED_OBJECT', msg)
                    track_alrt[obj['id']] = 'ABANDONED'

        # ══════════════════════════════════════════
        # 7. PERSON NEAR PLATFORM EDGE  (metro)
        # ══════════════════════════════════════════
        edge_pct = rules.get('platform_edge_zone_pct')
        if edge_pct:
            edge_y = int(frame_h * edge_pct)
            for t in persons:
                if t['center'][1] >= edge_y:
                    alert_key = f'edge_{t["id"]}'
                    if self._can_alert(alert_key, frame_idx):
                        msg = f'PERSON NEAR TRACK EDGE #{t["id"]}'
                        alerts.append(msg)
                        self._log(scene, frame_idx, 'PLATFORM_EDGE', msg)
                    track_alrt[t['id']] = 'EDGE DANGER'

        # ══════════════════════════════════════════
        # 8. PERSON FALLEN  (metro — aspect ratio check)
        # ══════════════════════════════════════════
        fall_thresh = rules.get('fall_aspect_thresh')
        if fall_thresh:
            for t in persons:
                if t['aspect'] >= fall_thresh:
                    alert_key = f'fall_{t["id"]}'
                    if self._can_alert(alert_key, frame_idx):
                        msg = f'PERSON MAY HAVE FALLEN #{t["id"]} (aspect={t["aspect"]})'
                        alerts.append(msg)
                        self._log(scene, frame_idx, 'FALLEN_PERSON', msg)
                    track_alrt[t['id']] = 'FALLEN'

        # ══════════════════════════════════════════
        # 9. SUDDEN CROWD DISPERSAL  (crowd / metro)
        # ══════════════════════════════════════════
        if rules.get('sudden_disperse') and len(self.person_count_history) >= 15:
            recent_avg  = np.mean(list(self.person_count_history)[-5:])
            earlier_avg = np.mean(list(self.person_count_history)[:10])
            if earlier_avg > 5 and recent_avg < earlier_avg * 0.45:
                if self._can_alert('disperse', frame_idx):
                    msg = f'SUDDEN DISPERSAL — crowd dropped from {earlier_avg:.0f} to {recent_avg:.0f}'
                    alerts.append(msg)
                    self._log(scene, frame_idx, 'SUDDEN_DISPERSAL', msg)

        # ══════════════════════════════════════════
        # 10. VEHICLE IN PEDESTRIAN ZONE  (street)
        # ══════════════════════════════════════════
        if rules.get('vehicle_in_ped_zone'):
            # Simple heuristic: vehicle detected in a scene type that should only have people
            for t in vehicles:
                alert_key = f'vedped_{t["id"]}'
                if self._can_alert(alert_key, frame_idx):
                    msg = f'VEHICLE IN PED ZONE — {t["name"]} #{t["id"]}'
                    alerts.append(msg)
                    self._log(scene, frame_idx, 'VEHICLE_IN_PED_ZONE', msg)
                track_alrt[t['id']] = 'PED ZONE!'

        # ══════════════════════════════════════════
        # 11. VEHICLE SPEEDING IN PARKING LOT
        # ══════════════════════════════════════════
        speed_thresh = rules.get('vehicle_speeding_px', 9999)
        for t in vehicles:
            if t['speed'] >= speed_thresh:
                alert_key = f'speed_{t["id"]}'
                if self._can_alert(alert_key, frame_idx):
                    msg = f'SPEEDING {t["name"]} #{t["id"]} — {t["speed"]:.1f}px/f'
                    alerts.append(msg)
                    self._log(scene, frame_idx, 'VEHICLE_SPEEDING', msg)
                track_alrt[t['id']] = 'SPEEDING'

        return alerts, track_alrt

    def _log(self, scene, frame_idx, alert_type, msg):
        self.alert_log.append({
            'scene':       scene,
            'frame':       frame_idx,
            'time_sec':    round(frame_idx / 25, 2),
            'alert_type':  alert_type,
            'message':     msg,
            'timestamp':   pd.Timestamp.now().strftime('%H:%M:%S'),
        })

    def get_log_df(self):
        return pd.DataFrame(self.alert_log)


print('✅ AnomalyDetector loaded — 11 anomaly rules across 5 scene types.')

---
## 📢 CELL 6 — Alert System (Jupyter Popup + Log + Summary)
Visual alerts inside the notebook with IPython display.

In [ ]:
from IPython.display import display, HTML, clear_output
import time

# ─────────────────────────────────────────────────────
# Color severity levels
# ─────────────────────────────────────────────────────
ALERT_SEVERITY = {
    'OVERCROWDING':       ('🔴 CRITICAL', '#ff2222', '#400000'),
    'TRAFFIC_JAM':        ('🟠 HIGH',     '#ff8800', '#3a1a00'),
    'STOPPED_VEHICLE':    ('🟡 MEDIUM',   '#ffcc00', '#2e2200'),
    'RUNNING':            ('🟠 HIGH',     '#ff6600', '#2e1000'),
    'LOITERING':          ('🟡 MEDIUM',   '#ddcc00', '#2a2000'),
    'ABANDONED_OBJECT':   ('🔴 CRITICAL', '#ff3300', '#3a0800'),
    'PLATFORM_EDGE':      ('🔴 CRITICAL', '#ff0055', '#3a0015'),
    'FALLEN_PERSON':      ('🔴 CRITICAL', '#ff0000', '#400000'),
    'SUDDEN_DISPERSAL':   ('🔴 CRITICAL', '#ff0099', '#3a0030'),
    'VEHICLE_IN_PED_ZONE':('🟠 HIGH',     '#ff9900', '#3a2000'),
    'VEHICLE_SPEEDING':   ('🟠 HIGH',     '#ff7700', '#3a1500'),
}


def jupyter_alert(scene, frame_idx, alert_type, message):
    """
    Show a styled HTML alert box inside the Jupyter cell output.
    """
    label, fg_color, bg_color = ALERT_SEVERITY.get(
        alert_type, ('⚠️ ALERT', '#ffff00', '#222200')
    )
    ts = time.strftime('%H:%M:%S')

    html = f"""
    <div style="
        font-family: monospace;
        background: {bg_color};
        border: 2px solid {fg_color};
        border-left: 6px solid {fg_color};
        border-radius: 6px;
        padding: 10px 16px;
        margin: 4px 0;
        color: {fg_color};
        display: flex;
        align-items: center;
        gap: 16px;
    ">
      <span style="font-size:1.4em">{label.split()[0]}</span>
      <div>
        <strong style="font-size:0.95em">
          {label.split(' ',1)[1]}  —  {alert_type.replace('_',' ')}
        </strong><br/>
        <span style="color:#ccc; font-size:0.82em">
          Scene: <b style="color:{fg_color}">{scene.upper()}</b> &nbsp;|
          Frame: <b>{frame_idx}</b> &nbsp;|
          Time: <b>{ts}</b>
        </span><br/>
        <span style="color:#eee; font-size:0.88em; margin-top:3px; display:block">
          {message}
        </span>
      </div>
    </div>
    """
    display(HTML(html))


def show_alert_summary(detector, scene):
    """
    Print a formatted summary after processing a full video.
    """
    log_df = detector.get_log_df()
    if log_df.empty:
        display(HTML('<div style="color:lime;font-family:monospace;padding:8px">✅ No anomalies detected in this video.</div>'))
        return log_df

    scene_df  = log_df[log_df['scene'] == scene]
    type_cnts = scene_df['alert_type'].value_counts().to_dict()

    rows = ''
    for atype, cnt in sorted(type_cnts.items(), key=lambda x: -x[1]):
        label, fg, bg = ALERT_SEVERITY.get(atype, ('⚠️ ALERT', '#ffff00', '#222200'))
        rows += f"""
        <tr>
          <td style="color:{fg};font-weight:bold">{label}</td>
          <td style="color:#eee">{atype.replace('_',' ')}</td>
          <td style="color:{fg};font-weight:bold;text-align:center">{cnt}</td>
          <td style="color:#aaa;font-size:0.85em">
            {scene_df[scene_df["alert_type"]==atype].iloc[-1]["message"]}
          </td>
        </tr>"""

    html = f"""
    <div style="font-family:monospace; background:#111; border-radius:10px;
                padding:16px; margin:10px 0; border:1px solid #333">
      <h3 style="color:#61dafb; margin:0 0 10px">
        🚨 Alert Summary — {scene.upper()} — {len(scene_df)} total alerts
      </h3>
      <table style="width:100%; border-collapse:collapse">
        <tr style="background:#1e2130; color:#888; font-size:0.8em">
          <th style="padding:6px;text-align:left">Severity</th>
          <th style="padding:6px;text-align:left">Alert Type</th>
          <th style="padding:6px;text-align:center">Count</th>
          <th style="padding:6px;text-align:left">Last Message</th>
        </tr>
        {rows}
      </table>
    </div>
    """
    display(HTML(html))
    return scene_df


print('✅ Alert display system ready — jupyter_alert() + show_alert_summary() loaded.')

---
## 🚗 CELL 7 — Master Run Function
Single reusable function — called by all 5 scene cells below.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from IPython.display import display, HTML, Image as IPImage
from ultralytics import YOLO
import time, os
from tqdm.notebook import tqdm

# Load YOLO model ONCE (reused for all scenes)
print('📥 Loading YOLOv8 model...')
yolo_model = YOLO(YOLO_MODEL)
print(f'✅ Model loaded: {YOLO_MODEL}')


def run_tracking(scene_name, video_path,
                 show_preview_every=50,
                 save_output=True):
    """
    Full pipeline for one video:
      1. Read frames
      2. Run YOLOv8 + ByteTrack
      3. Detect anomalies
      4. Draw overlays
      5. Save annotated video
      6. Show inline previews + alerts
    """
    if not os.path.exists(video_path):
        display(HTML(f'<div style="color:red;font-family:monospace">'
                     f'❌ Video not found: {video_path}<br>'
                     f'Update VIDEO_PATHS in Cell 3 and re-run.</div>'))
        return None

    # ── Setup ──
    cap     = cv2.VideoCapture(video_path)
    fps_in  = cap.get(cv2.CAP_PROP_FPS) or 25
    W       = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    H       = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_f = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    proc_f  = min(total_f, MAX_FRAMES) if MAX_FRAMES else total_f

    out_path = os.path.join(OUTPUT_DIR, f'{scene_name}_tracked.mp4')
    writer   = None
    if save_output:
        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        writer = cv2.VideoWriter(out_path, fourcc, fps_in, (W, H))

    tracker  = ObjectTracker()
    detector = AnomalyDetector()

    # Statistics accumulation
    stats = {
        'frame_ids':     [],
        'human_counts':  [],
        'vehicle_counts':[],
        'object_counts': [],
        'alert_counts':  [],
        'unique_ids':    set(),
    }

    preview_frames = []   # store a few frames to show inline
    all_alerts     = []   # running list for banner

    display(HTML(f"""
    <div style="font-family:monospace; background:#0d1117; border:1px solid #30363d;
                border-radius:8px; padding:12px; color:#58a6ff">
      🎬 Processing <b>{scene_name.upper()}</b> — {video_path}<br/>
      <span style="color:#8b949e">
        {W}×{H} | {fps_in:.0f}fps | {proc_f} frames | {proc_f/fps_in:.1f}s
      </span>
    </div>"""))

    t_start    = time.time()
    frame_idx  = 0

    for fi in tqdm(range(proc_f), desc=f'  {scene_name}', ncols=80):
        ret, frame = cap.read()
        if not ret:
            break

        # Skip frames for speed
        if fi % PROCESS_EVERY_N != 0:
            if writer: writer.write(frame)
            continue

        frame_idx += 1

        # ── YOLOv8 detection + ByteTrack ──
        results = yolo_model.track(
            frame,
            persist=True,           # keeps track IDs consistent across frames
            classes=TRACK_CLASS_IDS,
            conf=CONF_THRESHOLD,
            iou=IOU_THRESHOLD,
            tracker='bytetrack.yaml',
            verbose=False
        )

        # ── Parse tracking results ──
        tracks = tracker.update(results, frame_idx)

        # ── Anomaly detection ──
        frame_alerts, track_alert_map = detector.detect(
            tracks, tracker, frame, scene_name, frame_idx, H, W
        )
        all_alerts.extend(frame_alerts)

        # ── Jupyter alerts (print to notebook as they happen) ──
        for alrt in frame_alerts:
            atype = alrt.split('—')[0].strip().replace(' ','_')
            jupyter_alert(scene_name, frame_idx, atype.upper(), alrt)

        # ── Draw everything ──
        draw_trails(frame, tracker)
        for t in tracks:
            is_alert  = t['id'] in track_alert_map
            alert_msg = track_alert_map.get(t['id'], '')
            draw_box(frame, t, alert=is_alert, alert_msg=alert_msg)

        # Counts
        n_humans   = sum(1 for t in tracks if t['category']=='human')
        n_vehicles = sum(1 for t in tracks if t['category']=='vehicle')
        n_objects  = sum(1 for t in tracks if t['category']=='object')

        elapsed     = time.time() - t_start
        fps_display = frame_idx / max(elapsed, 0.001)

        draw_hud(frame,
                 {'human': n_humans, 'vehicle': n_vehicles, 'object': n_objects},
                 all_alerts[-10:], fps_display, scene_name, frame_idx)
        draw_alert_banner(frame, all_alerts[-5:])

        # Stats tracking
        stats['frame_ids'].append(frame_idx)
        stats['human_counts'].append(n_humans)
        stats['vehicle_counts'].append(n_vehicles)
        stats['object_counts'].append(n_objects)
        stats['alert_counts'].append(len(frame_alerts))
        for t in tracks:
            stats['unique_ids'].add(t['id'])

        # Save preview frames for inline display
        if frame_idx % show_preview_every == 0:
            preview_frames.append(cv2.cvtColor(frame.copy(), cv2.COLOR_BGR2RGB))

        if writer:
            writer.write(frame)

    cap.release()
    if writer:
        writer.release()

    # ── Inline preview grid ──
    if preview_frames:
        n    = min(len(preview_frames), 6)
        cols = 3
        rows = (n + cols - 1) // cols
        fig, axes = plt.subplots(rows, cols,
                                  figsize=(cols * 5, rows * 3.2))
        axes = axes.flatten() if hasattr(axes, 'flatten') else [axes]
        for i in range(len(axes)):
            if i < n:
                axes[i].imshow(preview_frames[i])
                axes[i].axis('off')
                axes[i].set_title(f'Frame sample {i+1}', fontsize=9)
            else:
                axes[i].axis('off')
        plt.suptitle(f'🎬 {scene_name.upper()} — Tracking Preview Frames',
                     fontsize=12, fontweight='bold')
        plt.tight_layout()
        plt.savefig(os.path.join(OUTPUT_DIR, f'{scene_name}_preview.png'),
                    dpi=100, bbox_inches='tight')
        plt.show()

    # ── Summary stats inline ──
    elapsed_total = time.time() - t_start
    display(HTML(f"""
    <div style="font-family:monospace; background:#161b22; border:1px solid #30363d;
                border-radius:8px; padding:14px; margin:8px 0">
      <b style="color:#58a6ff">✅ {scene_name.upper()} — Processing Complete</b><br/>
      <span style="color:#8b949e">
        Frames processed: <b style="color:#e6edf3">{frame_idx}</b> &nbsp;|
        Time: <b style="color:#e6edf3">{elapsed_total:.1f}s</b> &nbsp;|
        Avg FPS: <b style="color:#e6edf3">{frame_idx/elapsed_total:.1f}</b><br/>
        Unique IDs tracked: <b style="color:#e6edf3">{len(stats['unique_ids'])}</b> &nbsp;|
        Total alerts: <b style="color:#f85149">{len(all_alerts)}</b> &nbsp;|
        Output: <b style="color:#3fb950">{out_path if save_output else 'Not saved'}</b>
      </span>
    </div>"""))

    # ── Alert summary table ──
    show_alert_summary(detector, scene_name)

    # Save alert log CSV
    log_df = detector.get_log_df()
    if not log_df.empty:
        csv_path = os.path.join(OUTPUT_DIR, f'{scene_name}_alerts.csv')
        log_df.to_csv(csv_path, index=False)

    return stats, detector


print('✅ run_tracking() function ready. Now run Cells 8–12 to process each scene.')

---
## 🚗 CELL 8 — Run on TRAFFIC Video
Tracks: Cars · Trucks · Buses · Motorcycles  
Alerts: Traffic jam · Stopped vehicle · Pedestrian in road · Wrong-way

In [ ]:
display(HTML("""
<div style="background:#1a0a00;border-left:5px solid #ff8800;
            padding:10px 16px;border-radius:6px;font-family:monospace;color:#ff8800">
  🚗 SCENE: TRAFFIC  — monitoring for jams · stopped vehicles · pedestrians in road
</div>"""))

traffic_stats, traffic_detector = run_tracking(
    scene_name    = 'traffic',
    video_path    = VIDEO_PATHS['traffic'],
    show_preview_every = 40,
    save_output   = True,
)

---
## 👥 CELL 9 — Run on CROWD Video
Tracks: People  
Alerts: Overcrowding · Running · Sudden dispersal (panic)

In [ ]:
display(HTML("""
<div style="background:#000f1a;border-left:5px solid #0099ff;
            padding:10px 16px;border-radius:6px;font-family:monospace;color:#0099ff">
  👥 SCENE: CROWD  — monitoring for overcrowding · panic · running people
</div>"""))

crowd_stats, crowd_detector = run_tracking(
    scene_name    = 'crowd',
    video_path    = VIDEO_PATHS['crowd'],
    show_preview_every = 40,
    save_output   = True,
)

---
## 🅿️ CELL 10 — Run on PARKING LOT Video
Tracks: Cars · Trucks · People · Bags  
Alerts: Loitering · Abandoned bag · Speeding vehicle

In [ ]:
display(HTML("""
<div style="background:#0a1a00;border-left:5px solid #44cc00;
            padding:10px 16px;border-radius:6px;font-family:monospace;color:#44cc00">
  🅿️ SCENE: PARKING LOT  — monitoring for loitering · abandoned objects · speeding
</div>"""))

parking_stats, parking_detector = run_tracking(
    scene_name    = 'parking',
    video_path    = VIDEO_PATHS['parking'],
    show_preview_every = 40,
    save_output   = True,
)

---
## 🚇 CELL 11 — Run on METRO / STATION Video
Tracks: People  
Alerts: Platform edge · Fallen person · Crowd crush · Running

In [ ]:
display(HTML("""
<div style="background:#1a001a;border-left:5px solid #cc00ff;
            padding:10px 16px;border-radius:6px;font-family:monospace;color:#cc00ff">
  🚇 SCENE: METRO STATION  — monitoring for edge danger · falls · overcrowding
</div>"""))

metro_stats, metro_detector = run_tracking(
    scene_name    = 'metro',
    video_path    = VIDEO_PATHS['metro'],
    show_preview_every = 40,
    save_output   = True,
)

---
## 🚶 CELL 12 — Run on STREET Video
Tracks: People · Vehicles · Bags  
Alerts: Vehicle in pedestrian zone · Loitering · Abandoned object · Running

In [ ]:
display(HTML("""
<div style="background:#1a1a00;border-left:5px solid #ffff00;
            padding:10px 16px;border-radius:6px;font-family:monospace;color:#cccc00">
  🚶 SCENE: STREET  — monitoring for vehicle intrusion · loitering · abandoned bags
</div>"""))

street_stats, street_detector = run_tracking(
    scene_name    = 'street',
    video_path    = VIDEO_PATHS['street'],
    show_preview_every = 40,
    save_output   = True,
)

---
## 📊 CELL 13 — Full Analytics Dashboard
Cross-scene comparison: object counts over time, alert breakdown, heatmaps.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import numpy as np
import pandas as pd
import os

# ── Gather all results ──
all_scene_stats = {
    'traffic': traffic_stats   if 'traffic_stats'  in dir() and traffic_stats  else None,
    'crowd':   crowd_stats     if 'crowd_stats'    in dir() and crowd_stats    else None,
    'parking': parking_stats   if 'parking_stats'  in dir() and parking_stats  else None,
    'metro':   metro_stats     if 'metro_stats'    in dir() and metro_stats    else None,
    'street':  street_stats    if 'street_stats'   in dir() and street_stats   else None,
}
all_detectors = {
    'traffic': traffic_detector  if 'traffic_detector'  in dir() else None,
    'crowd':   crowd_detector    if 'crowd_detector'    in dir() else None,
    'parking': parking_detector  if 'parking_detector'  in dir() else None,
    'metro':   metro_detector    if 'metro_detector'    in dir() else None,
    'street':  street_detector   if 'street_detector'   in dir() else None,
}

# Only include scenes that were actually run
active_scenes = {k: v for k, v in all_scene_stats.items() if v is not None}

if not active_scenes:
    print('⚠️  No scenes processed yet. Run Cells 8–12 first.')
else:
    scene_colors = {
        'traffic':'#FF8800','crowd':'#0099FF','parking':'#44CC00',
        'metro':'#CC00FF','street':'#CCCC00'
    }

    fig = plt.figure(figsize=(22, 18))
    fig.patch.set_facecolor('#0d1117')
    fig.suptitle('🎯 Smart Surveillance — Full Analytics Dashboard',
                 fontsize=16, fontweight='bold', color='white', y=0.98)
    gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.35)

    # ── Plot 1: Human counts over frames per scene ──
    ax1 = fig.add_subplot(gs[0, :])
    ax1.set_facecolor('#161b22')
    for scene, stats in active_scenes.items():
        ax1.plot(stats['frame_ids'], stats['human_counts'],
                 label=scene.capitalize(), color=scene_colors[scene], linewidth=1.5, alpha=0.85)
    ax1.set_title('🧍 Person Count Over Time (all scenes)', color='white', fontsize=11)
    ax1.set_xlabel('Frame', color='#8b949e'); ax1.set_ylabel('Count', color='#8b949e')
    ax1.tick_params(colors='#8b949e'); ax1.legend(fontsize=9)
    for spine in ax1.spines.values(): spine.set_color('#30363d')

    # ── Plot 2: Vehicle counts over frames ──
    ax2 = fig.add_subplot(gs[1, :2])
    ax2.set_facecolor('#161b22')
    for scene, stats in active_scenes.items():
        if any(v > 0 for v in stats['vehicle_counts']):
            ax2.plot(stats['frame_ids'], stats['vehicle_counts'],
                     label=scene.capitalize(), color=scene_colors[scene], linewidth=1.5)
    ax2.set_title('🚗 Vehicle Count Over Time', color='white', fontsize=11)
    ax2.set_xlabel('Frame', color='#8b949e'); ax2.set_ylabel('Count', color='#8b949e')
    ax2.tick_params(colors='#8b949e'); ax2.legend(fontsize=9)
    for spine in ax2.spines.values(): spine.set_color('#30363d')

    # ── Plot 3: Alert counts per scene (bar chart) ──
    ax3 = fig.add_subplot(gs[1, 2])
    ax3.set_facecolor('#161b22')
    all_alert_counts = {}
    for scene, det in all_detectors.items():
        if det is not None:
            all_alert_counts[scene] = len(det.get_log_df())
    if all_alert_counts:
        bars = ax3.bar(all_alert_counts.keys(),
                       all_alert_counts.values(),
                       color=[scene_colors.get(s,'gray') for s in all_alert_counts],
                       edgecolor='#30363d', width=0.6)
        for bar, val in zip(bars, all_alert_counts.values()):
            ax3.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
                     str(val), ha='center', color='white', fontsize=10, fontweight='bold')
    ax3.set_title('🚨 Total Alerts per Scene', color='white', fontsize=11)
    ax3.set_ylabel('Alert Count', color='#8b949e')
    ax3.tick_params(colors='#8b949e')
    for spine in ax3.spines.values(): spine.set_color('#30363d')

    # ── Plot 4: Alert type breakdown stacked bar ──
    ax4 = fig.add_subplot(gs[2, :2])
    ax4.set_facecolor('#161b22')
    all_logs = []
    for scene, det in all_detectors.items():
        if det and not det.get_log_df().empty:
            all_logs.append(det.get_log_df())
    if all_logs:
        combined = pd.concat(all_logs, ignore_index=True)
        pivot    = combined.groupby(['scene','alert_type']).size().unstack(fill_value=0)
        alert_colors_list = plt.cm.tab20(np.linspace(0, 1, len(pivot.columns)))
        pivot.plot(kind='bar', stacked=True, ax=ax4, color=alert_colors_list,
                   edgecolor='none', width=0.6)
        ax4.set_title('🔍 Alert Types per Scene (stacked)', color='white', fontsize=11)
        ax4.set_xlabel('Scene', color='#8b949e')
        ax4.set_ylabel('Count', color='#8b949e')
        ax4.tick_params(colors='#8b949e', axis='x', rotation=0)
        ax4.tick_params(colors='#8b949e', axis='y')
        ax4.legend(fontsize=7, ncol=2, loc='upper right',
                   facecolor='#161b22', labelcolor='white', edgecolor='#30363d')
        for spine in ax4.spines.values(): spine.set_color('#30363d')

    # ── Plot 5: Alert timeline for each scene ──
    ax5 = fig.add_subplot(gs[2, 2])
    ax5.set_facecolor('#161b22')
    if all_logs:
        for scene, det in all_detectors.items():
            if det and not det.get_log_df().empty:
                df_log = det.get_log_df()
                ax5.scatter(df_log['frame'], [scene]*len(df_log),
                            color=scene_colors.get(scene,'gray'),
                            s=20, alpha=0.7, marker='|')
    ax5.set_title('⏱ Alert Timeline (when alerts occurred)', color='white', fontsize=10)
    ax5.set_xlabel('Frame Number', color='#8b949e')
    ax5.tick_params(colors='#8b949e')
    for spine in ax5.spines.values(): spine.set_color('#30363d')

    plt.savefig(os.path.join(OUTPUT_DIR, 'dashboard.png'),
                dpi=120, bbox_inches='tight', facecolor='#0d1117')
    plt.show()

    # ── Combined CSV ──
    if all_logs:
        combined.to_csv(os.path.join(OUTPUT_DIR, 'all_alerts_combined.csv'), index=False)
        display(HTML(f"""
        <div style="font-family:monospace;background:#0d1117;border:1px solid #30363d;
                    border-radius:8px;padding:12px;color:#3fb950">
          📁 All outputs saved to: {OUTPUT_DIR}<br/>
          &nbsp;├── traffic_tracked.mp4<br/>
          &nbsp;├── crowd_tracked.mp4<br/>
          &nbsp;├── parking_tracked.mp4<br/>
          &nbsp;├── metro_tracked.mp4<br/>
          &nbsp;├── street_tracked.mp4<br/>
          &nbsp;├── *_alerts.csv  (per scene)<br/>
          &nbsp;├── all_alerts_combined.csv<br/>
          &nbsp;└── dashboard.png
        </div>"""))

    print(f'\n🎉 Dashboard complete!')

---
## 🎥 CELL 14 — Run on YOUR OWN Custom Video
Point to any video file and choose a scene type.

In [ ]:
# ── Edit these two lines ──────────────────────────────────
MY_VIDEO = r'D:\AI FOR SOCIAL GOOD 1\videos\my_custom_video.mp4'
MY_SCENE = 'street'   # choose: traffic | crowd | parking | metro | street
# ─────────────────────────────────────────────────────────

custom_stats, custom_detector = run_tracking(
    scene_name         = MY_SCENE,
    video_path         = MY_VIDEO,
    show_preview_every = 30,
    save_output        = True,
)